In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
from tqdm import tqdm
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
#warnings.filterwarnings('ignore')
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')

In [3]:
start = dt.datetime(2019,6,20)
start1 = str(start)
end = dt.datetime(2019,6,26)
end1 = str(end)
print(start,end)

2019-06-20 00:00:00 2019-06-26 00:00:00


In [4]:
app_version = '2.0.2'

# new user reference

In [5]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query = (
    f"""SELECT
    * FROM 
    `hitwicketsuperstars.analytics_190927423.new_user_reference`"""
)
new_user_reference = client.query(query).to_dataframe()

In [6]:
new_user_reference['user_first_touch_timestamp'] = new_user_reference['user_first_touch_timestamp'].astype('datetime64[s]')
android_new = new_user_reference[(new_user_reference['user_first_touch_timestamp'] >= start)]

# Ftue completed users

In [7]:
query = (f"""
             SELECT 
             user_id as device_id,
             event_timestamp
             FROM `hitwicketsuperstars.analytics_190927423.events_*`,
             UNNEST(event_params) AS params    
             WHERE _TABLE_SUFFIX BETWEEN "{start1.replace('-','')}"
             AND "{end1.replace('-','')}"
             AND app_info.version = '{app_version}'
            AND device.operating_system = 'ANDROID'
             AND params.key = 'action'
            AND event_name IN ('natasha')
           AND params.value.string_value = 'hand_pointer_achievements_clicked'""")
         
ftue_complete = client.query(query).to_dataframe() 

In [8]:
ftue_complete.columns = ['device_id','create_time']
ftue_complete.sort_values('create_time',inplace=True,ascending=False)
ftue_complete.drop_duplicates('device_id',inplace=True)
ftue_complete = ftue_complete[ftue_complete['device_id'].isin(android_new['device_id'])]
ftue_complete['create_time'] = pd.to_datetime(ftue_complete['create_time'], unit = 'us')
print(len(ftue_complete))
ftue_complete.head()

673


,device_id,create_time
640,602f46bae4430f13fa5f1d7d18a70644,2019-06-26 18:09:09.186007
623,efcced8f07408ab9b357213e1bad9d6c,2019-06-26 17:17:31.192007
654,9fd654fc07acf3187139049d032a5b31,2019-06-26 17:16:45.178008
607,08aeabee748b7666d24a831b7718b670,2019-06-26 17:03:33.647008
634,3bc49bdf96d184d85d15f08b5e22f1b3,2019-06-26 16:59:10.874007


In [9]:
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$gte': start}},{"sign_up_details",'login_details.last_request_at'}): # end condition
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
users = pd.DataFrame(dic_flattened)
users = users[["_id","sign_up_details_device_id",'login_details_last_request_at']]
users.columns = ["user_id","device_id",'last_request']
len(users)

2258

In [10]:
users.sort_values(['device_id','last_request'],ascending=False,inplace=True)
users.drop_duplicates('device_id',inplace=True)
print(len(users))

2197


In [11]:
ftue_complete_user = pd.merge(ftue_complete,users,on='device_id')
ftue_complete_user = ftue_complete_user[['user_id','device_id','create_time']]

In [12]:
print(len(ftue_complete_user))
ftue_complete_user.head()

673


,user_id,device_id,create_time
0,5d13b4587119e60013b3e9d6,602f46bae4430f13fa5f1d7d18a70644,2019-06-26 18:09:09.186007
1,5d1249a7c5ebf8001185851f,efcced8f07408ab9b357213e1bad9d6c,2019-06-26 17:17:31.192007
2,5d13a7f8114e23001a9596e9,9fd654fc07acf3187139049d032a5b31,2019-06-26 17:16:45.178008
3,5d13a4b4114e23001a954a35,08aeabee748b7666d24a831b7718b670,2019-06-26 17:03:33.647008
4,5d1112c4fa80e30028d2b431,3bc49bdf96d184d85d15f08b5e22f1b3,2019-06-26 16:59:10.874007


# number of users who have trained

In [13]:
query = (f"""
             SELECT user_id as device_id,
                    event_timestamp
             FROM `hitwicketsuperstars.analytics_190927423.events_*`,
             UNNEST(event_params) AS params    
             WHERE _TABLE_SUFFIX BETWEEN "{start1.replace('-','')}"
             AND "{end1.replace('-','')}"
            AND params.key = 'action'
             AND app_info.version = '{app_version}'
          AND params.value.string_value IN ('sent_to_training','sent_to_training_using_hitcoins')
           
         """)
df = client.query(query).to_dataframe()

In [14]:
df['event_timestamp'] = pd.to_datetime(df['event_timestamp'],unit='us')

In [15]:
training_users = pd.merge(ftue_complete,df,on='device_id')
training_users = training_users[training_users['event_timestamp']>training_users['create_time']]
training_users = training_users[(training_users['event_timestamp']-training_users['create_time'])<'24:00:00']
training_users.drop_duplicates('device_id',inplace=True)

In [16]:
training_users = pd.merge(training_users,users,on='device_id')
training_users['d1'] =(training_users['last_request']-training_users['create_time']) > '24:00:00'

# number of users who have used end now

In [17]:
c_end_training = cursor.superstars.user_collectables_logs
aw_end_training_coins = []
for documents in c_end_training.aggregate([{'$unwind':"$data"},   # used for ending training once the training has started
                    {"$match" : {"data.reason_type" : 'END_TRAINING', # shown as 'FINISH' right next to 'SPEEDUP', once the training starts
                                  "type":"HARD_CURRENCY",               
                                  'data.quantity': {'$lt': 0},           
                                  'data.created_at': {'$gte': start}}}]):
    aw_end_training_coins.append(documents)
    
dic_flattened = [flatten(d) for d in aw_end_training_coins]
end_training_coins = pd.DataFrame(dic_flattened)
end_training_coins = end_training_coins[end_training_coins["user"].isin(training_users['user_id'])]
end_training_coins = end_training_coins[["_id","data_created_at",'user']]
end_training_coins.columns = ["end_now_coin_id", "end_now_coin_used_at","user_id"]

In [18]:
print(len(end_training_coins))
end_training_coins.head()

183


,end_now_coin_id,end_now_coin_used_at,user_id
289,5d0d8e7e8185c200192b7294,2019-06-22 02:12:14.253,5d0d8cd9fa80e30028307158
290,5d0d8e7e8185c200192b7294,2019-06-22 02:16:52.466,5d0d8cd9fa80e30028307158
292,5d0d9d458185c200192d2878,2019-06-22 07:59:34.060,5d0d9a98fa80e3002831dcd0
297,5d0da3c9fa80e3002832a3b1,2019-06-22 03:43:05.424,5d0da343fa80e3002832a1d3
298,5d0dabb28185c200192f1de0,2019-06-22 04:18:37.638,5d0daab0fa80e3002833ba6e


In [19]:
instant_users = pd.merge(ftue_complete_user,end_training_coins,on='user_id')
instant_users = instant_users[instant_users['end_now_coin_used_at']-instant_users['create_time']<'24:00:00']
len(instant_users)

147

In [20]:
grouped = instant_users.groupby('user_id').agg({'end_now_coin_id':'count'}).reset_index()
print(len(grouped))
grouped.head()

82


,user_id,end_now_coin_id
0,5d0d8cd9fa80e30028307158,2
1,5d0d9197fa80e30028311af8,1
2,5d0d9a98fa80e3002831dcd0,1
3,5d0da343fa80e3002832a1d3,1
4,5d0daab0fa80e3002833ba6e,1


# planning and grouping

In [21]:
grouped_zero = pd.merge(grouped,training_users[['user_id','d1']],on='user_id',how='right')
grouped_zero = grouped_zero.fillna(0)
len(grouped_zero)

334

In [22]:
grouped_zero.head()

,user_id,end_now_coin_id,d1
0,5d0d8cd9fa80e30028307158,2.0,False
1,5d0d9197fa80e30028311af8,1.0,False
2,5d0d9a98fa80e3002831dcd0,1.0,True
3,5d0da343fa80e3002832a1d3,1.0,False
4,5d0daab0fa80e3002833ba6e,1.0,False


In [23]:
distribution = grouped_zero.groupby('end_now_coin_id').agg({'user_id':'count','d1':'sum'})

In [24]:
distribution['user_cumsum'] = distribution.loc[::-1,'user_id'].cumsum()[::-1]
distribution['d1_cumsum'] = distribution.loc[::-1,'d1'].cumsum()[::-1]

In [25]:
distribution

,user_id,d1,user_cumsum,d1_cumsum
end_now_coin_id,,,,
0.0,252,65.0,334,91.0
1.0,58,15.0,82,26.0
2.0,12,5.0,24,11.0
3.0,2,1.0,12,6.0
4.0,4,1.0,10,5.0
5.0,2,2.0,6,4.0
6.0,1,0.0,4,2.0
8.0,2,1.0,3,2.0
11.0,1,1.0,1,1.0
